In [16]:
# Dependencies
import pandas as pd
import os
import re
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders

# Base Directory
base_dir = os.path.abspath("/Users/amberteetsel/MSDS/NLP/nlp-author-identification/")

The tokenizer wrapper (`BPETokenizer` class) is original code that provides a simplified interface for the pipeline. The underlying BPE model, pre-tokenizers, and trainer are Hugging Face's `tokenizers` library implementations, used as instructed in the assignment (not reimplemented).

In [2]:
class BPETokenizer:
    """Wraps HuggingFace's tokenizers library to provide a simple
    encode/decode/train interface for n-gram LM pipeline."""

    def __init__(self, vocab_size=5000, pre_tokenizer="whitespace"):
        self.vocab_size = vocab_size
        self.tokenizer = Tokenizer(models.BPE(unk_token="<unk>"))

        if pre_tokenizer == "whitespace":
            self.tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()
        elif pre_tokenizer == "byte_level":
            self.tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel()
        else:
            raise ValueError(f"Unknown pre_tokenizer: {pre_tokenizer}")

        self.special_tokens = ["<unk>", "<pad>", "<s>", "</s>"]

    def train(self, filepaths):
        """Train BPE on one or more text files (paths as a list)."""
        trainer = trainers.BpeTrainer(
            vocab_size=self.vocab_size,
            special_tokens=self.special_tokens,
        )
        self.tokenizer.train(filepaths, trainer)

    def encode(self, text):
        """Returns list of token ids."""
        return self.tokenizer.encode(text).ids

    def decode(self, ids):
        """Returns string from list of token ids."""
        return self.tokenizer.decode(ids)

    def get_vocab_size(self):
        return self.tokenizer.get_vocab_size()

    def save(self, path):
        self.tokenizer.save(path)

    def load(self, path):
        self.tokenizer = Tokenizer.from_file(path)

In [3]:
# Training
hobbit_train_path = os.path.join(base_dir, "texts", "hobbit_train.txt")
lostworld_train_path = os.path.join(base_dir, "texts", "lostworld_train.txt")

bpe = BPETokenizer(vocab_size=5000, pre_tokenizer="whitespace")
bpe.train([hobbit_train_path, lostworld_train_path])

In [4]:
sample = "Bilbo Baggins was a hobbit who lived in a hole in the ground."
ids = bpe.encode(sample)
print(ids)
print(bpe.decode(ids))
print("vocab size:", bpe.get_vocab_size())

[240, 723, 117, 55, 475, 285, 1230, 88, 55, 1161, 88, 87, 685, 12]
Bilbo Baggins was a hobbit who lived in a hole in the ground .
vocab size: 5000


In [5]:
sample = """
Well, at least
you are better than that herd of swine in Vienna, whose gregarious
grunt is, however, not more offensive than the isolated effort of the
British hog.
"""
ids = bpe.encode(sample)
print(ids)
print(bpe.decode(ids))
print("vocab size:", bpe.get_vocab_size())

[673, 10, 94, 980, 132, 201, 913, 356, 126, 1051, 58, 99, 1040, 426, 88, 4601, 3288, 10, 2161, 225, 1855, 741, 255, 2651, 105, 10, 1015, 10, 161, 276, 3635, 356, 87, 4123, 3168, 99, 87, 2613, 3568, 159, 61, 12]
Well , at least you are better than that her d of sw ine in Vien na , whose gre gar ious gr unt is , however , not more offensive than the isolated effort of the Br itish ho g .
vocab size: 5000


The decode is inserting a space between every token, even subword pieces that should be merged (e.g. "her d" instead of "herd"). This is the default behavior of the `Whitespace()` pre-tokenizer, so we'll move to the `ByteLevel` implementation.

After one iteration of `ByteLevel`, we discovered that any character that happened to not appear in training (e.g. "\n") will fall back to `<unk>` and cause incorrect spacing (merging words that should be separate). To fix, we specify that the trainer should start out with all 256 byte tokens for alphabet up front, instead of only what it happens to see in training corpus.

In [17]:
class BPETokenizer:
    def __init__(self, vocab_size=5000, pre_tokenizer="byte_level"):
        self.vocab_size = vocab_size
        self.tokenizer = Tokenizer(models.BPE(unk_token="<unk>"))

        if pre_tokenizer == "byte_level":
            self.tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)
            self.tokenizer.decoder = decoders.ByteLevel()
        elif pre_tokenizer == "whitespace":
            self.tokenizer.pre_tokenizer = pre_tokenizers.Whitespace()
            # no matching decoder so use default
        else:
            raise ValueError(f"Unknown pre_tokenizer: {pre_tokenizer}")

        self.special_tokens = ["<unk>", "<pad>", "<s>", "</s>"]

    def train(self, filepaths):
        """Train BPE on one or more text files (paths as a list)."""
        trainer = trainers.BpeTrainer(
            vocab_size=self.vocab_size,
            special_tokens=self.special_tokens,
            initial_alphabet=pre_tokenizers.ByteLevel.alphabet()
        )
        self.tokenizer.train(filepaths, trainer)

    def encode(self, text):
        """Normalize input and returns list of token ids."""
        text = re.sub(r"\s+", " ", text).strip()
        return self.tokenizer.encode(text).ids

    def decode(self, ids):
        """Returns string from list of token ids."""
        return self.tokenizer.decode(ids)

    def get_vocab_size(self):
        return self.tokenizer.get_vocab_size()

    def save(self, path):
        self.tokenizer.save(path)

    def load(self, path):
        self.tokenizer = Tokenizer.from_file(path)

In [18]:
# Training
hobbit_train_path = os.path.join(base_dir, "texts", "hobbit_train.txt")
lostworld_train_path = os.path.join(base_dir, "texts", "lostworld_train.txt")

bpe = BPETokenizer(vocab_size=5000, pre_tokenizer="byte_level")
bpe.train([hobbit_train_path, lostworld_train_path])

In [19]:
sample = "Bilbo Baggins was a hobbit who lived in a hole in the ground."
ids = bpe.encode(sample)
print(ids)
print(bpe.decode(ids))
print("vocab size:", bpe.get_vocab_size())

[4680, 982, 309, 262, 734, 545, 1569, 296, 262, 1982, 296, 263, 1102, 17]
Bilbo Baggins was a hobbit who lived in a hole in the ground.
vocab size: 5000


In [20]:
sample = """
Well, at least
you are better than that herd of swine in Vienna, whose gregarious
grunt is, however, not more offensive than the isolated effort of the
British hog.
"""
ids = bpe.encode(sample)
print(ids)
print(bpe.decode(ids))
print("vocab size:", bpe.get_vocab_size())

[1089, 15, 366, 1282, 332, 468, 1205, 604, 325, 1564, 71, 281, 661, 671, 296, 1583, 2149, 3747, 15, 2562, 753, 2721, 776, 536, 407, 87, 379, 15, 1309, 15, 360, 515, 615, 3655, 604, 263, 4675, 3317, 281, 263, 3233, 4368, 481, 74, 17]
Well, at least you are better than that herd of swine in Vienna, whose gregarious grunt is, however, not more offensive than the isolated effort of the British hog.
vocab size: 5000
